In [1]:
# =============================================================================
# STEP 9 (nb49) - LABEL-FREE SUBTYPE MIXTURE ESTIMATION
#
# This is a GATE. Everything downstream depends on it, so it is built to fail
# loudly and early rather than to be salvaged.
#
# WHY. The placebo ladder (Section 5.4) showed that per-subtype coverage is
# essentially fixed and that class-level coverage is the mixture-weighted average
# of those fixed components: warezmaster carries 92 per cent of the NSL trend
# because it is the only component whose coverage differs. That is not merely a
# diagnosis, it is a MODEL of the failure. The source quantile is calibrated for
# the source subtype mixture P_s and applied to data drawn from a different target
# mixture P_t. If P_t can be estimated WITHOUT target labels, the source
# calibration scores can be reweighted by P_t(t)/P_s(t) before the quantile is
# taken, and the quantile becomes correct for the target composition.
#
# THIS NOTEBOOK ONLY ESTIMATES P_t. It does not reweight anything. If the estimate
# is poor, the method is dead and we stop after one notebook.
#
# METHOD. Black-box shift estimation (Lipton et al., 2018): train a subtype
# classifier on the source, apply it to unlabelled target flows, then invert the
# source confusion matrix to correct the predicted distribution:
#
#     C q = p_hat      where C[i,j] = P(predict i | true j) on held-out source
#
# solved for q under a simplex constraint. Consistent under label shift, which is
# what a change of subtype mixture at fixed per-subtype behaviour IS.
#
# THE KNOWN CEILING, STATED BEFORE ANY RESULT. Eight of fourteen R2L target
# subtypes have ZERO source instances: snmpguess (11.5 per cent of target mass),
# snmpgetattack (6.2), httptunnel (4.6) and five smaller. No estimator trained on
# the source can name a class it has never seen, so at best this recovers the
# mixture over the SIX supported subtypes, covering 76.6 per cent of target R2L
# mass. The remaining 23.4 per cent is unreachable by construction and the
# notebook measures it rather than hiding it.
# =============================================================================
import numpy as np, pandas as pd
from scipy import stats
from scipy.optimize import nnls
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
import os, sys, json, shutil, glob, subprocess, hashlib, time
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT=Path('/content/drive/MyDrive'); PARENT_DIR=DRIVE_ROOT/'CALSHIFT_Research'
PROJECT_ROOT=PARENT_DIR/'calshift-research'; CRED_DIR=DRIVE_ROOT/'.gitcreds'
assert PROJECT_ROOT.exists(), 'Drive mount unhealthy; restart runtime and remount'
subprocess.run(['git','config','--global','user.name','Md Anas Biswas'],check=False)
subprocess.run(['git','config','--global','user.email','anasbiswas@gmail.com'],check=False)
subprocess.run(['git','config','--global','credential.helper','store'],check=False)
for fn,dest in [('.git-credentials','/root/.git-credentials'),('.gitconfig','/root/.gitconfig')]:
    for cand in (PARENT_DIR/fn, CRED_DIR/fn):
        if cand.exists(): shutil.copy(cand,dest); os.chmod(dest,0o600); break
os.chdir(PROJECT_ROOT); sys.path.insert(0,str(PROJECT_ROOT/'src'))
import importlib
for m in ['config','conformal']:
    if m in sys.modules: importlib.reload(sys.modules[m])
import config
RD=config.REPORTS_DIR; ALPHA=config.ALPHA_PRIMARY
MIN_SRC=3          # a subtype needs at least this many source points to be estimable
print('ready')


Mounted at /content/drive
ready


In [2]:
# =============================================================================
# Cell 2 - the estimator. Confusion-corrected label-shift estimation, with the
# simplex constraint enforced by non-negative least squares rather than by
# clipping a possibly-negative matrix inverse.
# =============================================================================
def confusion_matrix_cv(X, y, K, seed=0, folds=5):
    """C[i,j] = P(predict i | true j), estimated out-of-fold so it is not optimistic."""
    C=np.zeros((K,K)); counts=np.zeros(K)
    kf=StratifiedKFold(min(folds, np.bincount(y, minlength=K)[np.bincount(y, minlength=K)>0].min()),
                       shuffle=True, random_state=seed)
    for tr,te in kf.split(X,y):
        m=RandomForestClassifier(n_estimators=300, min_samples_leaf=1, class_weight='balanced',
                                 n_jobs=-1, random_state=seed).fit(X[tr], y[tr])
        pred=m.predict(X[te])
        for t,p in zip(y[te], pred): C[p,t]+=1
        for t in y[te]: counts[t]+=1
    for j in range(K):
        if counts[j]>0: C[:,j]/=counts[j]
    return C

def estimate_mixture(C, p_hat):
    """Solve C q = p_hat for q on the simplex. NNLS keeps q >= 0; renormalise for sum 1."""
    q,_=nnls(C, p_hat)
    s=q.sum()
    return q/s if s>1e-12 else np.full(len(q), 1.0/len(q))

def tv(a,b): return 0.5*float(np.abs(np.asarray(a)-np.asarray(b)).sum())

print('estimator defined')
print('  C is estimated OUT OF FOLD on source data, so it is not optimistic')
print('  q is solved on the simplex by NNLS, not by inverting a possibly ill-conditioned C')


estimator defined
  C is estimated OUT OF FOLD on source data, so it is not optimistic
  q is solved on the simplex by NNLS, not by inverting a possibly ill-conditioned C


In [3]:
# =============================================================================
# Cell 3 - NSL-KDD R2L. The hardest case and the one the method must survive.
# =============================================================================
CL=config.CANONICAL_CLASSES
tr=pd.read_parquet(config.INTERIM_DIR/'nslkdd_train.parquet').reset_index(drop=True)
te=pd.read_parquet(config.INTERIM_DIR/'nslkdd_test.parquet').reset_index(drop=True)
part=pd.read_parquet(config.PROC_DIR/'nslkdd_source_partition_labels.parquet')
tr=tr.assign(partition=part['partition'].values)
DROP={'label','subtype','partition','is_unseen'}
FE=[c for c in tr.columns if c not in DROP and pd.api.types.is_numeric_dtype(tr[c])]

src=tr[(tr.partition=='source_cal_pool')&(tr.label=='R2L')]
tgt=te[te.label=='R2L']
print(f"R2L source calibration pool: {len(src)} flows | target: {len(tgt)} flows")

src_counts=src['subtype'].value_counts()
SUP=[s for s in src_counts.index if src_counts[s]>=MIN_SRC]   # estimable subtypes
print(f"\nsubtypes with >= {MIN_SRC} source instances (estimable): {SUP}")
tgt_all=tgt['subtype'].value_counts()
unreachable=[s for s in tgt_all.index if s not in SUP]
mass_unreachable=float(tgt_all[unreachable].sum()/tgt_all.sum())
print(f"target subtypes NOT estimable: {unreachable}")
print(f"  they carry {mass_unreachable:.1%} of target R2L mass -- unreachable by construction")

s2i={s:i for i,s in enumerate(SUP)}; K=len(SUP)
m_src=src['subtype'].isin(SUP)
Xs=src.loc[m_src, FE].to_numpy(float); ys=src.loc[m_src,'subtype'].map(s2i).to_numpy()
Xt=tgt[FE].to_numpy(float)
print(f"\ntraining the subtype classifier on {len(Xs)} source flows across {K} subtypes")
print("  per-subtype source counts:", {s:int((ys==i).sum()) for s,i in s2i.items()})

C=confusion_matrix_cv(Xs, ys, K, seed=0)
print("\nout-of-fold confusion matrix C[predict, true]:")
print(pd.DataFrame(C, index=[f'pred {s}' for s in SUP], columns=SUP).round(3).to_string())
diag=np.diag(C)
print(f"\n  per-subtype recall: {dict(zip(SUP, diag.round(3)))}")
print(f"  condition number of C: {np.linalg.cond(C):.1f}   (large means the inversion is unstable)")


R2L source calibration pool: 149 flows | target: 2885 flows

subtypes with >= 3 source instances (estimable): ['warezclient', 'guess_passwd', 'warezmaster', 'ftp_write']
target subtypes NOT estimable: ['snmpguess', 'snmpgetattack', 'httptunnel', 'multihop', 'named', 'sendmail', 'xlock', 'xsnoop', 'phf', 'imap']
  they carry 24.5% of target R2L mass -- unreachable by construction

training the subtype classifier on 146 source flows across 4 subtypes
  per-subtype source counts: {'warezclient': 129, 'guess_passwd': 11, 'warezmaster': 3, 'ftp_write': 3}

out-of-fold confusion matrix C[predict, true]:
                   warezclient  guess_passwd  warezmaster  ftp_write
pred warezclient           1.0         0.000          0.0      0.667
pred guess_passwd          0.0         0.909          0.0      0.000
pred warezmaster           0.0         0.000          1.0      0.000
pred ftp_write             0.0         0.091          0.0      0.333

  per-subtype recall: {'warezclient': np.float64(

In [4]:
# =============================================================================
# Cell 4 - estimate the target mixture and compare against the truth the method
# never sees.
# =============================================================================
mdl=RandomForestClassifier(n_estimators=300, min_samples_leaf=1, class_weight='balanced',
                           n_jobs=-1, random_state=0).fit(Xs, ys)
pred_t=mdl.predict(Xt)
p_hat=np.array([(pred_t==i).mean() for i in range(K)])
q_hat=estimate_mixture(C, p_hat)

# ground truth, restricted to the estimable subtypes and renormalised
true_counts=np.array([int((tgt['subtype']==s).sum()) for s in SUP], float)
q_true = true_counts/true_counts.sum() if true_counts.sum()>0 else np.zeros(K)
p_src   = np.array([(ys==i).mean() for i in range(K)])

R=pd.DataFrame({'subtype':SUP,'source_share':p_src,'raw_predicted':p_hat,
                'estimated':q_hat,'true':q_true})
R['abs_error']=(R.estimated-R['true']).abs()
print("MIXTURE ESTIMATION, restricted to estimable subtypes")
print(R.round(4).to_string(index=False))
print()
print(f"  total variation, estimated vs true : {tv(q_hat,q_true):.4f}")
print(f"  total variation, raw predicted     : {tv(p_hat,q_true):.4f}   (uncorrected baseline)")
print(f"  total variation, source vs true    : {tv(p_src,q_true):.4f}   (doing nothing)")
print(f"  max per-subtype absolute error     : {R.abs_error.max():.4f}")

print("\nGATE")
improved = tv(q_hat,q_true) < tv(p_src,q_true)
beats_raw = tv(q_hat,q_true) <= tv(p_hat,q_true) + 0.02
usable   = tv(q_hat,q_true) < 0.25
print(f"  better than assuming no shift : {improved}")
print(f"  at least as good as raw counts: {beats_raw}")
print(f"  total variation under 0.25    : {usable}")
if improved and usable:
    print("\n  GATE PASSED. The target mixture is recoverable without labels well enough")
    print("  to build reweighted calibration on. Proceed to nb50.")
else:
    print("\n  GATE FAILED. The mixture cannot be recovered accurately enough, so weights")
    print("  built from it would be noise. The method stops here and that is reported as")
    print("  a negative result rather than patched.")


MIXTURE ESTIMATION, restricted to estimable subtypes
     subtype  source_share  raw_predicted  estimated   true  abs_error
 warezclient        0.8836         0.9418     0.9411 0.0000     0.9411
guess_passwd        0.0753         0.0423     0.0464 0.5652     0.5188
 warezmaster        0.0205         0.0125     0.0125 0.4334     0.4210
   ftp_write        0.0205         0.0035     0.0000 0.0014     0.0014

  total variation, estimated vs true : 0.9411
  total variation, raw predicted     : 0.9439   (uncorrected baseline)
  total variation, source vs true    : 0.9027   (doing nothing)
  max per-subtype absolute error     : 0.9411

GATE
  better than assuming no shift : False
  at least as good as raw counts: True
  total variation under 0.25    : False

  GATE FAILED. The mixture cannot be recovered accurately enough, so weights
  built from it would be noise. The method stops here and that is reported as
  a negative result rather than patched.


In [5]:
# =============================================================================
# Cell 5 - the weights this would produce, and their effective sample size.
# This is where the method's boundary becomes visible.
# =============================================================================
w = np.where(p_src>1e-12, q_hat/np.maximum(p_src,1e-12), 0.0)
W=pd.DataFrame({'subtype':SUP,'source_share':p_src,'est_target_share':q_hat,'weight':w})
W['n_source']=[int((ys==i).sum()) for i in range(K)]
print("IMPLIED CALIBRATION WEIGHTS")
print(W.round(4).to_string(index=False))

per_point = np.array([w[i] for i in ys])
ess = per_point.sum()**2 / np.maximum((per_point**2).sum(), 1e-12)
print(f"\n  raw calibration points        : {len(ys)}")
print(f"  effective sample size after reweighting: {ess:.1f}")
print(f"  ESS as a fraction of n        : {ess/len(ys):.3f}")
need = int(np.ceil(1/ALPHA))-1
print(f"  feasibility floor at alpha={ALPHA}: {need}")
if ess < need:
    print("\n  ESS FALLS BELOW THE FLOOR. Reweighting concentrates the calibration set onto")
    print("  a handful of points, so the weighted quantile would be unreliable even if the")
    print("  mixture estimate is perfect. This is the boundary predicted before running:")
    print("  the method cannot manufacture calibration data that does not exist.")
else:
    print("\n  ESS clears the floor; a weighted quantile is defensible on this class.")

# how much target mass the method can address at all
print(f"\n  target R2L mass in estimable subtypes : {1-mass_unreachable:.1%}")
print(f"  target R2L mass unreachable           : {mass_unreachable:.1%}")
print("  Even a perfect estimate leaves the unreachable share uncorrected, which bounds")
print("  the coverage recovery achievable on this class.")


IMPLIED CALIBRATION WEIGHTS
     subtype  source_share  est_target_share  weight  n_source
 warezclient        0.8836            0.9411  1.0652       129
guess_passwd        0.0753            0.0464  0.6159        11
 warezmaster        0.0205            0.0125  0.6069         3
   ftp_write        0.0205            0.0000  0.0000         3

  raw calibration points        : 146
  effective sample size after reweighting: 140.6
  ESS as a fraction of n        : 0.963
  feasibility floor at alpha=0.05: 19

  ESS clears the floor; a weighted quantile is defensible on this class.

  target R2L mass in estimable subtypes : 75.5%
  target R2L mass unreachable           : 24.5%
  Even a perfect estimate leaves the unreachable share uncorrected, which bounds
  the coverage recovery achievable on this class.


In [ ]:
# =============================================================================
# Cell 6 - save and commit.
# =============================================================================
out={'dataset':'nslkdd','class':'R2L','min_source_per_subtype':MIN_SRC,
     'estimable_subtypes':SUP,'unreachable_subtypes':unreachable,
     'unreachable_target_mass':float(mass_unreachable),
     'tv_estimated_vs_true':float(tv(q_hat,q_true)),
     'tv_raw_vs_true':float(tv(p_hat,q_true)),
     'tv_source_vs_true':float(tv(p_src,q_true)),
     'max_abs_error':float(R.abs_error.max()),
     'confusion_condition_number':float(np.linalg.cond(C)),
     'effective_sample_size':float(ess),'n_calibration':int(len(ys)),
     'feasibility_floor':int(need),
     'gate_passed':bool(improved and usable),
     'mixture':R.round(5).to_dict('records'),
     'weights':W.round(5).to_dict('records')}
(RD/'mixture_estimation_nslkdd.json').write_text(json.dumps(out, indent=2, default=str))
R.to_csv(RD/'mixture_estimation_nslkdd.csv', index=False)
W.to_csv(RD/'mixture_weights_nslkdd.csv', index=False)
print('saved mixture estimate, weights and the gate verdict')

def git(*a, show=True):
    r=subprocess.run(['git',*a],capture_output=True,text=True)
    if show and (r.stdout or r.stderr): print((r.stdout+r.stderr).strip())
    return r
for s,dd in [('/root/.git-credentials',PARENT_DIR/'.git-credentials'),('/root/.gitconfig',PARENT_DIR/'.gitconfig')]:
    if os.path.exists(s): shutil.copy(s,dd)
os.chdir(PROJECT_ROOT)
lock=PROJECT_ROOT/'.git'/'index.lock'
if lock.exists() and not subprocess.run(['pgrep','git'],capture_output=True).stdout.strip():
    lock.unlink(); print('removed stale git lock')
for attempt in (1,2):
    git('add','-A',show=False)
    if git('status','--porcelain',show=False).stdout.strip():
        git('commit','-m','nb49: label-free subtype mixture estimation; gate for mixture-matched conformal calibration')
        r=git('push','-u','origin','main')
        if r.returncode: print('PUSH FAILED. Commit is safe locally.')
        break
    if attempt==1: print('waiting 10s for Drive sync...'); time.sleep(10)
    else: print('nothing to commit')
print(git('log','--oneline','-3',show=False).stdout)


saved mixture estimate, weights and the gate verdict
removed stale git lock
